In [3]:
#matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler

train_set = pd.read_csv('data/train_set.csv')
features = ['laser energy', 'oxygen pressure', 'deposition_temp', 'TD_distance']
train_X = train_set[features]
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(train_X)
X_normalized = pd.DataFrame(X_normalized, columns=train_X.columns)
X_train = X_normalized
y_train = train_set['Jc_6']

In [4]:
class TreeNode:
    def __init__(self, feature_idx=None, left_bound=None, right_bound=None, 
                 left=None, right=None, value=None, a_opt=None, 
                 combo_feat_idx=None, raw_candidate_feat=None):
        self.feature_idx = feature_idx                
        self.left_bound = left_bound                  
        self.right_bound = right_bound                
        self.left = left                              
        self.right = right                            
        self.value = value                            
        self.a_opt = a_opt                            
        self.combo_feat_idx = combo_feat_idx          
        self.raw_candidate_feat = raw_candidate_feat  

class CART:
    def __init__(self, task="classification", max_depth=5, min_samples_split=2, lambda_=0.1):
        self.task = task
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.lambda_ = lambda_       
        self.root = None             
        self.init_n_features = None  

    def _gini(self, y):
        if len(y) == 0:
            return 0
        y = y.astype(int)
        class_counts = np.bincount(y)
        probabilities = class_counts / len(y)
        gini = 1 - np.sum(probabilities ** 2)
        return gini

    def _mse(self, y):
        if len(y) == 0:
            return 0
        return np.mean((y - np.mean(y)) ** 2)

    def _base_loss(self, y):
        if self.task == "classification":
            return self._gini(y)
        else:
            return self._mse(y)

    def _get_linear_weights(self, X, y, candidate_feat, combo_feat_idx):
        used_feature_indices = [combo_feat_idx, candidate_feat]
        X_used = X[:, used_feature_indices]
        
        ridge = Ridge(alpha=self.lambda_, fit_intercept=False)
        try:
            ridge.fit(X_used, y)
            return ridge.coef_
        except:
            return None

    def _best_split(self, X, y, depth, remaining_raw_features, latest_combo_feat_idx):
        n_samples, n_features = X.shape
        best_feature_idx = None 
        best_left_bound = None
        best_right_bound = None
        best_a_opt = None               
        best_gain = -np.inf             
        base_loss = self._base_loss(y)  

        if depth == 0:
            for feature_idx in range(self.init_n_features):  
                X_feature = X[:, feature_idx].copy()
                unique_vals = np.unique(X_feature)
                sorted_vals = np.sort(unique_vals)

                for i in range(len(sorted_vals)):
                    left_bound = sorted_vals[i]
                    for j in range(i+1, len(sorted_vals)):
                        right_bound = sorted_vals[j]
                        in_mask = (X_feature >= left_bound) & (X_feature <= right_bound)
                        out_mask = ~in_mask
                        y_in, y_out = y[in_mask], y[out_mask]

                        if len(y_in) == 0 or len(y_out) == 0:
                            continue

                        if self.task == "classification":
                            loss = (len(y_in)*self._gini(y_in) + len(y_out)*self._gini(y_out)) / len(y)
                        else:
                            loss = (len(y_in)*self._mse(y_in) + len(y_out)*self._mse(y_out)) / len(y)

                        current_gain = base_loss - loss

                        if current_gain > best_gain:
                            best_gain = current_gain
                            best_feature_idx = feature_idx
                            best_left_bound = left_bound
                            best_right_bound = right_bound

            return best_feature_idx, best_left_bound, best_right_bound, best_a_opt

        else:
            if not remaining_raw_features:           
                return None, None, None, None
            if latest_combo_feat_idx is None:        
                return None, None, None, None
            if latest_combo_feat_idx >= n_features:  
                return None, None, None, None

            for candidate_feat in remaining_raw_features:
                a_opt = self._get_linear_weights(X, y, candidate_feat, latest_combo_feat_idx)
                if a_opt is None:
                    continue

                used_feature_indices = [latest_combo_feat_idx, candidate_feat]
                X_used = X[:, used_feature_indices]
                proj_scores = X_used @ a_opt  

                unique_scores = np.unique(proj_scores)
                sorted_scores = np.sort(unique_scores)

                for i in range(len(sorted_scores)):
                    left_bound = sorted_scores[i]
                    for j in range(i+1, len(sorted_scores)):
                        right_bound = sorted_scores[j]
                        in_mask = (proj_scores >= left_bound) & (proj_scores <= right_bound)
                        out_mask = ~in_mask
                        y_in, y_out = y[in_mask], y[out_mask]

                        if len(y_in) == 0 or len(y_out) == 0:
                            continue

                        if self.task == "classification":
                            loss = (len(y_in)*self._gini(y_in) + len(y_out)*self._gini(y_out)) / len(y)
                        else:
                            loss = (len(y_in)*self._mse(y_in) + len(y_out)*self._mse(y_out)) / len(y)

                        current_gain = base_loss - loss

                        if current_gain > best_gain:
                            best_gain = current_gain
                            best_feature_idx = candidate_feat
                            best_left_bound = left_bound
                            best_right_bound = right_bound
                            best_a_opt = a_opt

            return best_feature_idx, best_left_bound, best_right_bound, best_a_opt

    def _leaf_value(self, y):
        if self.task == "classification":
            return np.argmax(np.bincount(y.astype(int))) 
        else:
            return np.mean(y)  

    def _append_combo_feat(self, X, proj_scores):
        X_new = np.hstack([X, proj_scores.reshape(-1, 1)])
        new_combo_feat_idx = X_new.shape[1] - 1
        return X_new, new_combo_feat_idx

    def _build_tree(self, X, y, depth, remaining_raw_features, latest_combo_feat_idx):
        n_samples = len(y)

        stop_conditions = [
            depth >= self.max_depth,
            n_samples < self.min_samples_split,
            self._base_loss(y) < 1e-6  
        ]
        if any(stop_conditions):
            return TreeNode(value=self._leaf_value(y))

        feat_idx, left_bound, right_bound, a_opt = self._best_split(
            X, y, depth, remaining_raw_features, latest_combo_feat_idx
        )
        if feat_idx is None:  
            return TreeNode(value=self._leaf_value(y))

        if depth == 0:
            X_feature = X[:, feat_idx].copy()
            in_mask = (X_feature >= left_bound) & (X_feature <= right_bound)
            out_mask = ~in_mask
            X_in, y_in = X[in_mask], y[in_mask]
            X_out, y_out = X[out_mask], y[out_mask]

            proj_scores_in = X_in[:, feat_idx]
            proj_scores_out = X_out[:, feat_idx]

            X_in_new, combo_idx_in = self._append_combo_feat(X_in, proj_scores_in)
            X_out_new, combo_idx_out = self._append_combo_feat(X_out, proj_scores_out)

            remaining_raw_in = [f for f in remaining_raw_features if f != feat_idx]
            remaining_raw_out = [f for f in remaining_raw_features if f != feat_idx]

            current_node = TreeNode(
                feature_idx=feat_idx,
                left_bound=left_bound,
                right_bound=right_bound,
                combo_feat_idx=latest_combo_feat_idx  
            )

        else:
            if a_opt is None or latest_combo_feat_idx is None:
                return TreeNode(value=self._leaf_value(y))

            used_feature_indices = [latest_combo_feat_idx, feat_idx]
            X_used = X[:, used_feature_indices]
            proj_scores = X_used @ a_opt

            in_mask = (proj_scores >= left_bound) & (proj_scores <= right_bound)
            out_mask = ~in_mask
            X_in, y_in = X[in_mask], y[in_mask]
            X_out, y_out = X[out_mask], y[out_mask]
            proj_scores_in = proj_scores[in_mask]
            proj_scores_out = proj_scores[out_mask]

            X_in_new, combo_idx_in = self._append_combo_feat(X_in, proj_scores_in)
            X_out_new, combo_idx_out = self._append_combo_feat(X_out, proj_scores_out)

            remaining_raw_in = [f for f in remaining_raw_features if f != feat_idx]
            remaining_raw_out = [f for f in remaining_raw_features if f != feat_idx]

            current_node = TreeNode(
                feature_idx=feat_idx,
                left_bound=left_bound,
                right_bound=right_bound,
                a_opt=a_opt,
                combo_feat_idx=latest_combo_feat_idx,
                raw_candidate_feat=feat_idx
            )

        left_node = self._build_tree(
            X_in_new, y_in, depth + 1, remaining_raw_in, combo_idx_in
        )
        right_node = self._build_tree(
            X_out_new, y_out, depth + 1, remaining_raw_out, combo_idx_out
        )

        current_node.left = left_node
        current_node.right = right_node

        return current_node

    def fit(self, X, y):
        X = np.array(X, dtype=np.float64)
        y = np.array(y)
        self.init_n_features = X.shape[1]
        remaining_raw_features = list(range(self.init_n_features))
        latest_combo_feat_idx = None  
        self.root = self._build_tree(
            X, y, depth=0,
            remaining_raw_features=remaining_raw_features,
            latest_combo_feat_idx=latest_combo_feat_idx
        )

    def _predict_sample(self, x, node):
        if node.value is not None:  
            return node.value

        feat_idx = node.feature_idx
        left_bound = node.left_bound
        right_bound = node.right_bound
        a_opt = node.a_opt
        combo_feat_idx = node.combo_feat_idx

        if a_opt is None:
            feat_val = x[feat_idx]
            if left_bound <= feat_val <= right_bound:
                x_new = np.hstack([x, np.array([feat_val])])
                return self._predict_sample(x_new, node.left)
            else:
                x_new = np.hstack([x, np.array([feat_val])])
                return self._predict_sample(x_new, node.right)

        else:
            if combo_feat_idx is None or combo_feat_idx >= len(x):  
                return self._predict_sample(x, node.right)

            used_feature_vals = np.array([x[combo_feat_idx], x[feat_idx]])
            proj_score = used_feature_vals @ a_opt
            x_new = np.hstack([x, np.array([proj_score])])
            if left_bound <= proj_score <= right_bound:
                return self._predict_sample(x_new, node.left)
            else:
                return self._predict_sample(x_new, node.right)

    def predict(self, X):
        X = np.array(X, dtype=np.float64)
        return np.array([self._predict_sample(x, self.root) for x in X])